In [4]:
import csv
import asyncio
import logging
from datetime import datetime
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv("C:/Users/nisha/Documents/student/.env")

PASS_MARKS = int(os.getenv("PASS_MARKS"))
FINE_PER_DAY = int(os.getenv("FINE_PER_DAY"))

Logging Setup

In [6]:
if not os.path.exists("C:/Users/nisha/Documents/student/logs"):
    os.makedirs("C:/Users/nisha/Documents/student/logs")
logging.basicConfig(
    filename="C:/Users/nisha/Documents/student/logs/errors.log",
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

Custom Exceptions

In [7]:
class InvalidMarksError(Exception):
    pass

class InvalidFeeError(Exception):
    pass

Decorator for Logging

In [8]:
def log_function(func):
    def wrapper(*args, **kwargs):
        print(f"Running {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

Student Class

In [9]:
class Student:
    def __init__(self, name, marks, fee_paid, late_days):
        self.name = name
        self.marks = int(marks)
        self.fee_paid = int(fee_paid)
        self.late_days = int(late_days)

    def check_pass(self):
        return self.marks >= PASS_MARKS

    def calculate_fine(self):
        return self.late_days * FINE_PER_DAY

Load CSV

In [10]:
def load_students(file):
    students = []
    with open(file, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                marks = int(row["marks"])
                fee = int(row["fee_paid"])
                if marks < 0 or marks > 100:
                    raise InvalidMarksError("Marks must be 0-100")
                student = Student(
                    row["name"],
                    marks,
                    fee,
                    row["late_days"]
                )
                students.append(student)
            except Exception as e:
                logging.error(str(e))
    return students

Async Process & Generate Report

In [11]:
async def process_student(student):
    await asyncio.sleep(0.1)
    status = "PASS" if student.check_pass() else "FAIL"
    fine = student.calculate_fine()
    return f"{student.name} - {status} - Fine: {fine}"

@log_function
async def generate_report(students):
    tasks = [process_student(s) for s in students]
    results = await asyncio.gather(*tasks)

    # Ensure reports folder exists
    if not os.path.exists("C:/Users/nisha/Documents/student/reports"):
        os.makedirs("C:/Users/nisha/Documents/student/reports")

    timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
    filename = f"C:/Users/nisha/Documents/student/reports/report_{timestamp}.txt"
    with open(filename, "w") as f:
        for r in results:
            f.write(r + "\n")
    print("Report generated:", filename)

In [18]:
async def run_portal():
    students = load_students("students.csv")
    await generate_report(students)
await run_portal()

Running generate_report
Report generated: C:/Users/nisha/Documents/student/reports/report_2026_03_08_20_48.txt
